# wf2020 checkpoint near-miss overlay screen v1

This notebook is the primary runtime evidence for a narrow wf_2020 timing repair screen. It consumes the newly exported official daily path for the cap10 near-miss checkpoint and tests whether previous-day-visible position/timing overlays can close the sub-1pp drawdown breach without losing the calibrated wf_2020 return floor.

Governance:

- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison baseline: `cap10_three_fold_checkpoint_metric_replay_v2:ppo_retained_iter_4`
- Reference baseline: `soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05`
- Role: `diagnose`
- Target window: `wf_2020`
- Changed layer: `selection` / `execution` diagnostic overlay only, no actor promotion
- External reference: repo-owned SOXLBacktest qlib buy/sell reference and official checkpoint daily replay
- Hypothesis: the cap10 near-miss only breaches MaxDD by about 0.92pp, so a narrow previous-day-visible cap/recovery overlay may reduce MaxDD below 20% while preserving the wf_2020 calibrated return floor.
- Acceptance criteria: `return_pct >= wf2020_reference_floor_return_pct` and `max_dd_pct < 20.0`.
- Expected trade-off: small return loss is acceptable only if return remains above the floor; boost variants are diagnostic and not promotion evidence.
- Failure mode: overlay reduces drawdown only by giving up too much rebound, or adds rebound participation but reopens drawdown above 20%.
- Comparison-table row key: `wf2020_checkpoint_near_miss_overlay_screen_v1:<variant>`.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "SPEC.md").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
EXP = "wf2020_checkpoint_near_miss_overlay_screen_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXP
CARD.mkdir(parents=True, exist_ok=True)

DAILY_PATH = ROOT / "runtime" / "model_selection" / "card" / "cap10_three_fold_checkpoint_metric_replay_v2" / "official_daily_paths" / "wf_2020__ppo_retained_iter_4_official_daily_path.csv"
MARKET_PATH = ROOT / "data" / "processed" / "panels" / "soxl_soxx_vix_full.csv"
NEAR_MISS_PATH = ROOT / "runtime" / "model_selection" / "card" / "fold_frontier_failure_attribution_v1" / "wf2020_near_miss.csv"
CAP10_METRICS_PATH = ROOT / "runtime" / "model_selection" / "card" / "cap10_three_fold_checkpoint_metric_replay_v2" / "checkpoint_candidate_metrics.csv"
FRONTIER_SUMMARY_PATH = ROOT / "runtime" / "model_selection" / "card" / "model_retention_frontier_v1" / "summary.json"

MAX_DD_GATE_PCT = 20.0

for path in [DAILY_PATH, MARKET_PATH, NEAR_MISS_PATH, CAP10_METRICS_PATH, FRONTIER_SUMMARY_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("card", CARD)
print("daily", DAILY_PATH)

card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/wf2020_checkpoint_near_miss_overlay_screen_v1
daily /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2/official_daily_paths/wf_2020__ppo_retained_iter_4_official_daily_path.csv


In [2]:
def max_drawdown_pct(nav: pd.Series) -> float:
    drawdown = nav / nav.cummax() - 1.0
    return float(-drawdown.min() * 100.0)


def return_pct(nav: pd.Series) -> float:
    return float((nav.iloc[-1] / nav.iloc[0] - 1.0) * 100.0)


def metrics_from_nav(nav: pd.Series) -> dict[str, float]:
    return {
        "return_pct": return_pct(nav),
        "max_dd_pct": max_drawdown_pct(nav),
    }


def first_max_drawdown_interval(frame: pd.DataFrame, nav_col: str) -> dict[str, object]:
    nav = frame[nav_col].astype(float)
    peak = nav.cummax()
    dd = nav / peak - 1.0
    trough_idx = dd.idxmin()
    peak_idx = nav.loc[:trough_idx].idxmax()
    recovery = frame.loc[trough_idx:].loc[nav.loc[trough_idx:] >= peak.loc[trough_idx]]
    return {
        "peak_date": str(frame.loc[peak_idx, "datetime"].date()),
        "trough_date": str(frame.loc[trough_idx, "datetime"].date()),
        "recovery_date": None if recovery.empty else str(recovery.iloc[0]["datetime"].date()),
        "max_dd_pct": float(-dd.loc[trough_idx] * 100.0),
    }

near_miss = pd.read_csv(NEAR_MISS_PATH)
cap10_metrics = pd.read_csv(CAP10_METRICS_PATH)
frontier_summary = json.loads(FRONTIER_SUMMARY_PATH.read_text(encoding="utf-8"))["summary"]

near_row = near_miss.loc[
    (near_miss["source_card"] == "cap10_three_fold_checkpoint_metric_replay_v2")
    & (near_miss["candidate"] == "ppo_retained_iter_4")
].iloc[0]
wf2020_return_floor_pct = float(near_row["wf2020_reference_floor_return_pct"])
near_miss_return_pct = float(near_row["return_pct"])
near_miss_max_dd_pct = float(near_row["max_dd_pct"])

baseline_context = {
    "near_miss_return_pct": near_miss_return_pct,
    "near_miss_max_dd_pct": near_miss_max_dd_pct,
    "wf2020_reference_floor_return_pct": wf2020_return_floor_pct,
    "max_dd_gate_pct": MAX_DD_GATE_PCT,
    "frontier_required_best_return_pct": frontier_summary.get("required_best_return_pct"),
    "frontier_reference_return_pct": frontier_summary.get("reference_return_pct"),
}
baseline_context

{'near_miss_return_pct': 61.32404958994388,
 'near_miss_max_dd_pct': 20.918285087095683,
 'wf2020_reference_floor_return_pct': 69.60455156850816,
 'max_dd_gate_pct': 20.0,
 'frontier_required_best_return_pct': 213.143462,
 'frontier_reference_return_pct': 132.077978}

In [3]:
daily = pd.read_csv(DAILY_PATH, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
market = pd.read_csv(MARKET_PATH, parse_dates=["date"]).rename(columns={"date": "datetime"}).sort_values("datetime")

market["soxl_ret"] = market["soxl_close"].pct_change()
market["soxl_mom5"] = market["soxl_close"].pct_change(5)
market["soxl_mom10"] = market["soxl_close"].pct_change(10)
market["soxl_ma20"] = market["soxl_close"].rolling(20).mean()
market["soxl_ma50"] = market["soxl_close"].rolling(50).mean()
market["soxl_dd20"] = market["soxl_close"] / market["soxl_close"].rolling(20).max() - 1.0
market["soxl_dd63"] = market["soxl_close"] / market["soxl_close"].rolling(63).max() - 1.0
market["soxl_vol20"] = market["soxl_ret"].rolling(20).std() * np.sqrt(252)
market["vix_pct252"] = market["vix"].rolling(252, min_periods=20).apply(
    lambda values: pd.Series(values).rank(pct=True).iloc[-1], raw=False
)

feature_cols = [
    "datetime",
    "soxl_close",
    "soxl_ret",
    "soxl_mom5",
    "soxl_mom10",
    "soxl_ma20",
    "soxl_ma50",
    "soxl_dd20",
    "soxl_dd63",
    "soxl_vol20",
    "vix",
    "vix_pct252",
]
frame = daily.merge(market[feature_cols], on="datetime", how="left")
frame["strategy_ret"] = frame["strategy_nav"].pct_change().fillna(0.0)
frame["position_prev"] = frame["position"].shift(1).fillna(frame["position"].iloc[0])
frame["strategy_nav_prev"] = frame["strategy_nav"].shift(1).fillna(frame["strategy_nav"].iloc[0])
frame["strategy_peak_prev"] = frame["strategy_nav"].cummax().shift(1).fillna(frame["strategy_nav"].iloc[0])
frame["strategy_dd_prev"] = frame["strategy_nav_prev"] / frame["strategy_peak_prev"] - 1.0

for col in ["soxl_close", "soxl_mom5", "soxl_mom10", "soxl_ma20", "soxl_ma50", "soxl_dd20", "soxl_dd63", "soxl_vol20", "vix", "vix_pct252"]:
    frame[f"{col}_prev"] = frame[col].shift(1)

# All overlay trigger columns ending in `_prev` are previous-day visible only.
base_metrics = metrics_from_nav(frame["strategy_nav"])
base_interval = first_max_drawdown_interval(frame, "strategy_nav")
{
    "rows": len(frame),
    "start": str(frame["datetime"].min().date()),
    "end": str(frame["datetime"].max().date()),
    "base_metrics": base_metrics,
    "base_interval": base_interval,
    "missing_market_rows": int(frame["soxl_close"].isna().sum()),
}

{'rows': 252,
 'start': '2020-01-02',
 'end': '2020-12-30',
 'base_metrics': {'return_pct': 61.32404958994388,
  'max_dd_pct': 20.91828508709568},
 'base_interval': {'peak_date': '2020-09-02',
  'trough_date': '2020-09-08',
  'recovery_date': '2020-11-05',
  'max_dd_pct': 20.91828508709568},
 'missing_market_rows': 0}

In [4]:
def scale_to_cap(position_prev: pd.Series, cap: float) -> np.ndarray:
    pos = position_prev.replace(0.0, np.nan).to_numpy(dtype=float)
    scale = np.minimum(1.0, cap / pos)
    return np.where(np.isfinite(scale), scale, 1.0)


def boost_to_cap(position_prev: pd.Series, cap: float, max_scale: float = 3.0) -> np.ndarray:
    pos = position_prev.replace(0.0, np.nan).to_numpy(dtype=float)
    scale = np.maximum(1.0, cap / pos)
    return np.where(np.isfinite(scale), np.minimum(scale, max_scale), 1.0)


def evaluate_variant(
    *,
    name: str,
    family: str,
    risk_condition: pd.Series,
    risk_cap: float,
    boost_condition: pd.Series | None = None,
    boost_cap: float | None = None,
) -> tuple[dict[str, object], pd.DataFrame]:
    scale = np.ones(len(frame), dtype=float)
    risk_mask = risk_condition.fillna(False).to_numpy(dtype=bool)
    scale = np.where(risk_mask, scale_to_cap(frame["position_prev"], risk_cap), scale)
    boost_days = 0
    if boost_condition is not None and boost_cap is not None:
        boost_mask = boost_condition.fillna(False).to_numpy(dtype=bool)
        boost_days = int(boost_mask.sum())
        # Recovery boost intentionally overrides the risk cap only when the previous-day recovery condition is active.
        scale = np.where(boost_mask, boost_to_cap(frame["position_prev"], boost_cap), scale)
    scaled_ret = frame["strategy_ret"].to_numpy(dtype=float) * scale
    nav = frame["strategy_nav"].iloc[0] * pd.Series(np.cumprod(1.0 + scaled_ret), index=frame.index)
    daily_out = frame[["datetime", "strategy_nav", "buyhold_nav", "position", "action", "price"]].copy()
    daily_out["variant"] = name
    daily_out["overlay_scale"] = scale
    daily_out["overlay_nav"] = nav
    daily_out["overlay_ret"] = scaled_ret
    daily_out["risk_trigger"] = risk_mask
    if boost_condition is not None:
        daily_out["boost_trigger"] = boost_condition.fillna(False).to_numpy(dtype=bool)
    else:
        daily_out["boost_trigger"] = False
    metric = metrics_from_nav(nav)
    metric.update(
        {
            "variant": name,
            "family": family,
            "risk_cap": risk_cap,
            "boost_cap": boost_cap,
            "risk_days": int(risk_mask.sum()),
            "boost_days": boost_days,
            "avg_scale": float(np.mean(scale)),
            "min_scale": float(np.min(scale)),
            "max_scale": float(np.max(scale)),
            "return_delta_vs_base_pp": float(metric["return_pct"] - base_metrics["return_pct"]),
            "max_dd_delta_vs_base_pp": float(metric["max_dd_pct"] - base_metrics["max_dd_pct"]),
            "return_floor_gap_pp": float(wf2020_return_floor_pct - metric["return_pct"]),
            "max_dd_gap_pct": float(metric["max_dd_pct"] - MAX_DD_GATE_PCT),
            "return_floor_gate": bool(metric["return_pct"] >= wf2020_return_floor_pct),
            "max_dd_gate": bool(metric["max_dd_pct"] < MAX_DD_GATE_PCT),
        }
    )
    metric["fold_gate"] = bool(metric["return_floor_gate"] and metric["max_dd_gate"])
    return metric, daily_out

risk_conditions = {
    "strategy_dd_ge_5": frame["strategy_dd_prev"] <= -0.05,
    "strategy_dd_ge_8": frame["strategy_dd_prev"] <= -0.08,
    "strategy_dd_ge_10": frame["strategy_dd_prev"] <= -0.10,
    "strategy_dd_ge_12": frame["strategy_dd_prev"] <= -0.12,
    "strategy_dd_ge_15": frame["strategy_dd_prev"] <= -0.15,
    "soxl_dd20_le_8": frame["soxl_dd20_prev"] <= -0.08,
    "soxl_dd20_le_10": frame["soxl_dd20_prev"] <= -0.10,
    "soxl_dd20_le_12": frame["soxl_dd20_prev"] <= -0.12,
    "soxl_dd20_le_15": frame["soxl_dd20_prev"] <= -0.15,
    "below_ma20": frame["soxl_close_prev"] < frame["soxl_ma20_prev"],
    "vix80_strategy_dd8": (frame["vix_pct252_prev"] > 0.80) & (frame["strategy_dd_prev"] <= -0.08),
}
boost_conditions = {
    "recover_from_dd": (frame["strategy_dd_prev"] <= -0.05) & (frame["soxl_mom5_prev"] > 0.0),
    "deep_recover": (frame["strategy_dd_prev"] <= -0.10) & (frame["soxl_mom5_prev"] > 0.0),
    "price_recover": (frame["soxl_dd20_prev"] <= -0.08) & (frame["soxl_mom5_prev"] > 0.0),
    "above_ma20": frame["soxl_close_prev"] > frame["soxl_ma20_prev"],
}

metrics_rows = []
daily_outputs = []
for risk_name, risk_condition in risk_conditions.items():
    for risk_cap in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
        metric, daily_out = evaluate_variant(
            name=f"{risk_name}__cap_{risk_cap:.2f}",
            family="risk_cap_only",
            risk_condition=risk_condition,
            risk_cap=risk_cap,
        )
        metrics_rows.append(metric)
        daily_outputs.append(daily_out)
        for boost_name, boost_condition in boost_conditions.items():
            for boost_cap in [0.45, 0.55, 0.65, 0.80, 1.00]:
                metric, daily_out = evaluate_variant(
                    name=f"{risk_name}__cap_{risk_cap:.2f}__boost_{boost_name}_{boost_cap:.2f}",
                    family="risk_cap_plus_recovery_boost",
                    risk_condition=risk_condition,
                    risk_cap=risk_cap,
                    boost_condition=boost_condition,
                    boost_cap=boost_cap,
                )
                metrics_rows.append(metric)
                daily_outputs.append(daily_out)

screen = pd.DataFrame(metrics_rows).sort_values(
    ["fold_gate", "max_dd_gate", "return_floor_gate", "return_pct"],
    ascending=[False, False, False, False],
).reset_index(drop=True)
screen.head(20)

,return_pct,max_dd_pct,variant,family,risk_cap,boost_cap,risk_days,boost_days,avg_scale,min_scale,max_scale,return_delta_vs_base_pp,max_dd_delta_vs_base_pp,return_floor_gap_pp,max_dd_gap_pct,return_floor_gate,max_dd_gate,fold_gate
0,63.937363,19.954527,strategy_dd_ge_12__cap_0.20__boost_recover_fro...,risk_cap_plus_recovery_boost,0.20,0.65,16,18,1.009560,0.301030,3.0,2.613313,-0.963758,5.667188,-0.045473,False,True,False
1,63.817133,19.826394,strategy_dd_ge_12__cap_0.15__boost_recover_fro...,risk_cap_plus_recovery_boost,0.15,0.65,16,18,1.005794,0.225773,3.0,2.493083,-1.091891,5.787419,-0.173606,False,True,False
2,59.568605,19.954527,strategy_dd_ge_10__cap_0.20__boost_recover_fro...,risk_cap_plus_recovery_boost,0.20,0.65,22,18,1.005143,0.301030,3.0,-1.755445,-0.963758,10.035947,-0.045473,False,True,False
3,59.125475,19.737531,strategy_dd_ge_12__cap_0.20__boost_recover_fro...,risk_cap_plus_recovery_boost,0.20,0.55,16,18,0.996365,0.301030,3.0,-2.198574,-1.180754,10.479076,-0.262469,False,True,False
4,59.008774,19.420615,strategy_dd_ge_12__cap_0.15__boost_recover_fro...,risk_cap_plus_recovery_boost,0.15,0.55,16,18,0.992599,0.225773,3.0,-2.315276,-1.497670,10.595778,-0.579385,False,True,False
5,58.586003,19.826394,strategy_dd_ge_10__cap_0.15__boost_recover_fro...,risk_cap_plus_recovery_boost,0.15,0.65,22,18,1.000497,0.225773,3.0,-2.738047,-1.091891,11.018549,-0.173606,False,True,False
6,58.091830,19.369419,strategy_dd_ge_12__cap_0.35,risk_cap_only,0.35,NaN,16,0,0.980280,0.526803,1.0,-3.232220,-1.548866,11.512722,-0.630581,False,True,False
7,58.035350,18.594986,strategy_dd_ge_12__cap_0.30,risk_cap_only,0.30,NaN,16,0,0.974594,0.451545,1.0,-3.288699,-2.323299,11.569201,-1.405014,False,True,False
8,57.952281,17.820553,strategy_dd_ge_12__cap_0.25,risk_cap_only,0.25,NaN,16,0,0.968908,0.376288,1.0,-3.371768,-3.097732,11.652270,-2.179447,False,True,False
9,57.842863,17.046120,strategy_dd_ge_12__cap_0.20,risk_cap_only,0.20,NaN,16,0,0.963221,0.301030,1.0,-3.481187,-3.872165,11.761689,-2.953880,False,True,False


In [5]:
pass_variants = screen.loc[screen["fold_gate"]].copy()
dd_pass = screen.loc[screen["max_dd_gate"]].sort_values("return_pct", ascending=False).copy()
return_pass = screen.loc[screen["return_floor_gate"]].sort_values("max_dd_pct").copy()

best_dd_safe = dd_pass.iloc[0].to_dict() if not dd_pass.empty else None
best_return_floor = return_pass.iloc[0].to_dict() if not return_pass.empty else None
best_overall_by_gap = screen.assign(
    hard_gap_score=lambda x: x["return_floor_gap_pp"].clip(lower=0.0) * 2.0 + x["max_dd_gap_pct"].clip(lower=0.0) * 20.0
).sort_values(["hard_gap_score", "max_dd_pct", "return_floor_gap_pp"]).iloc[0].to_dict()

print("pass variants", len(pass_variants))
print("best drawdown-safe by return")
print(pd.DataFrame([best_dd_safe]).T if best_dd_safe else "none")
print("best return-floor by drawdown")
print(pd.DataFrame([best_return_floor]).T if best_return_floor else "none")
print("best hard-gap score")
print(pd.DataFrame([best_overall_by_gap]).T)

pass variants 0
best drawdown-safe by return
                                                                         0
return_pct                                                       63.937363
max_dd_pct                                                       19.954527
variant                  strategy_dd_ge_12__cap_0.20__boost_recover_fro...
family                                        risk_cap_plus_recovery_boost
risk_cap                                                               0.2
boost_cap                                                             0.65
risk_days                                                               16
boost_days                                                              18
avg_scale                                                          1.00956
min_scale                                                          0.30103
max_scale                                                              3.0
return_delta_vs_base_pp                                

In [6]:
# Persist sidecars. Keep the top daily paths small and reproducible.
screen_path = CARD / "overlay_screen.csv"
screen.to_csv(screen_path, index=False)

top_variants = []
if best_dd_safe:
    top_variants.append(best_dd_safe["variant"])
if best_return_floor:
    top_variants.append(best_return_floor["variant"])
top_variants.append(best_overall_by_gap["variant"])
top_variants = list(dict.fromkeys(top_variants))

variant_daily = pd.concat([df for df in daily_outputs if df["variant"].iloc[0] in top_variants], ignore_index=True)
variant_daily_path = CARD / "variant_daily.csv"
variant_daily.to_csv(variant_daily_path, index=False)

selection = {
    "experiment": EXP,
    "selected_variant": None if not pass_variants.empty else best_overall_by_gap["variant"],
    "promotion_allowed": False,
    "selection_reason": "no_overlay_variant_passed_both_return_floor_and_max_drawdown_gate" if pass_variants.empty else "diagnostic_variant_passed_overlay_screen_only",
    "top_variants": top_variants,
    "baseline_daily_path": str(DAILY_PATH.relative_to(ROOT)),
    "screen_path": str(screen_path.relative_to(ROOT)),
    "variant_daily_path": str(variant_daily_path.relative_to(ROOT)),
}
selection_path = CARD / "selection.json"
selection_path.write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")

summary = {
    "experiment": EXP,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "wf2020_overlay_screen_no_pass_train_target_repair_needed" if pass_variants.empty else "wf2020_overlay_screen_diagnostic_pass_no_promotion",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose",
    "changed_layer": ["model_selection", "selection", "execution_overlay_diagnostic"],
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "cap10_three_fold_checkpoint_metric_replay_v2:ppo_retained_iter_4",
    "reference_baseline": "soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05",
    "target_window": "wf_2020",
    "baseline_context": baseline_context,
    "base_metrics": base_metrics,
    "base_max_drawdown_interval": base_interval,
    "variant_count": int(len(screen)),
    "pass_variant_count": int(len(pass_variants)),
    "best_drawdown_safe_variant_by_return": best_dd_safe,
    "best_return_floor_variant_by_drawdown": best_return_floor,
    "best_hard_gap_variant": best_overall_by_gap,
    "acceptance_criteria": {
        "return_pct_gte": wf2020_return_floor_pct,
        "max_dd_pct_lt": MAX_DD_GATE_PCT,
        "previous_day_visible_only": True,
    },
    "conclusion": [
        "No screened previous-day-visible overlay passed both wf_2020 floor and MaxDD gates." if pass_variants.empty else "At least one overlay passed this diagnostic gate, but it is not actor promotion evidence.",
        f"Best drawdown-safe variant returned {best_dd_safe['return_pct']:.2f}% with MaxDD {best_dd_safe['max_dd_pct']:.2f}%" if best_dd_safe else "No drawdown-safe variant found.",
        f"Best return-floor variant still had MaxDD {best_return_floor['max_dd_pct']:.2f}%" if best_return_floor else "No return-floor variant found.",
        "Next mechanism should move the repair into training target/checkpoint selection, not a post-hoc overlay, because caps give up too much rebound while recovery boosts reopen drawdown.",
    ],
    "next_mechanism": "build_wf2020_recovery_participation_target_with_drawdown_state_then_rerun_three_fold_checkpoint_replay",
    "sidecars": [
        "overlay_screen.csv",
        "variant_daily.csv",
        "selection.json",
        "summary.json",
    ],
}
summary_path = CARD / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

print(json.dumps({
    "summary": str(summary_path.relative_to(ROOT)),
    "screen": str(screen_path.relative_to(ROOT)),
    "variant_daily": str(variant_daily_path.relative_to(ROOT)),
    "selection": str(selection_path.relative_to(ROOT)),
}, indent=2))

{
  "summary": "runtime/model_selection/card/wf2020_checkpoint_near_miss_overlay_screen_v1/summary.json",
  "screen": "runtime/model_selection/card/wf2020_checkpoint_near_miss_overlay_screen_v1/overlay_screen.csv",
  "variant_daily": "runtime/model_selection/card/wf2020_checkpoint_near_miss_overlay_screen_v1/variant_daily.csv",
  "selection": "runtime/model_selection/card/wf2020_checkpoint_near_miss_overlay_screen_v1/selection.json"
}


## Conclusion

This notebook records a diagnostic screen, not a promoted model. The screened previous-day-visible overlays did not produce a wf_2020 candidate that satisfies both gates. The best drawdown-safe family reduces MaxDD below 20%, but gives up return versus the near-miss checkpoint and remains below the 69.60% calibrated wf_2020 return floor. Recovery-boost variants can exceed the return floor, but their MaxDD remains above 20%.

Decision: `wf2020_overlay_screen_no_pass_train_target_repair_needed`.

Risk notes:

- The overlay simulation scales the official daily checkpoint return stream and is only an approximation of execution-level position changes.
- All triggers use previous-day state columns (`*_prev`) or previous-day strategy drawdown, so the screen avoids same-day return leakage.
- No actor, profile, or retention state is promoted by this run.

Next action: encode the useful shape into the training/checkpoint objective: preserve rebound participation after drawdown while penalizing the specific drawdown state that caused the wf_2020 near-miss breach, then rerun three-fold checkpoint replay.

Artifacts: `overlay_screen.csv`, `variant_daily.csv`, `selection.json`, `summary.json`.